In [5]:
import requests
from bs4 import BeautifulSoup

In [16]:
def scrape_presidents(url: str = "https://en.wikipedia.org/wiki/List_of_presidents_of_the_United_States"):

    # Fetch the page and return None if error
    response = requests.get(url)
    if response.status_code != 200:
        print(f"ERROR: HTTP {response.status_code}")
        return None
    
    # Parse HTML
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Find the main presidents table
    # Look for the first <table class="wikitable"> after any heading with "Presidents"
    table = None
    for heading in soup.find_all(['h1', 'h2', 'h3']):
        if 'president' in heading.get_text().lower():
            table = heading.find_next("table", class_="wikitable")
            if table:
                break

    if not table:
        print("Could not find the presidents table.")
        return None

    # Extract rows
    rows = table.find_all("tr")
    presidents = []

    for row in rows[1:]:  # Skip header row
        cells = row.find_all(["td", "th"])
        if len(cells) < 6:  # Skip malformed rows
            continue

        def get_text(cell):
            return " ".join(cell.stripped_strings).strip()

        # Extract portrait image
        portrait = ""
        img = cells[1].find("img") if len(cells) > 1 else None
        if img and img.get("src"):
            portrait = "https:" + img["src"]

        president = {
            "no": get_text(cells[0]).rstrip('.'),  # Remove period
            "name": re.sub(r'\[.*?\]', '', get_text(cells[2])).strip(),  # Remove [a], [b]
            "term": get_text(cells[3]),
            "party": get_text(cells[4]),
            "election": get_text(cells[5]),
            "vice_president": get_text(cells[6]) if len(cells) > 6 else ""
        }
        if portrait:
            president["portrait"] = portrait

        presidents.append(president)

    # Build final data
    data = {
        "url": url,
        "title": "List of Presidents of the United States",
        "scraped_at": datetime.now().strftime("%d-%m-%Y %H:%M:%S"),
        "total_presidents": len(presidents),
        "presidents": presidents
    }

    # Save to JSON
    with open("22_Web_scraping/presidents.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print("Data successfully saved!")
    
    return data

In [17]:
scrape_presidents()

ERROR: HTTP 403
